# Marts

In [1]:
import sqlite3
import pandas as pd

query = """
    WITH roster_values AS (
        SELECT
            t.season,
            t.race_number,
            t.user_guid,
            t.team_no,
            COUNT(*) AS asset_count,
            COUNT(a.value) AS prices_n_found,
            COUNT(b.value) AS prices_next_found,
            ROUND(SUM(a.value), 1) AS value_feed_n,
            ROUND(SUM(b.value), 1) AS value_feed_next
        FROM team_race_asset t
        LEFT JOIN asset_race a
          ON a.season = t.season
         AND a.race_number = t.race_number
         AND a.asset_id = t.asset_id
        LEFT JOIN asset_race b
          ON b.season = t.season
         AND b.race_number = t.race_number + 1
         AND b.asset_id = t.asset_id
        GROUP BY
            t.season, t.race_number, t.user_guid, t.team_no
    )
    SELECT
        t.race_number,
        t.user_guid,
        t.team_no,
        t.team_name,
        t.recorded_asset_value,
        r.value_feed_n,
        r.value_feed_next,
        ROUND(
            r.value_feed_n - t.recorded_asset_value, 2
        ) AS difference_n,
        ROUND(
            r.value_feed_next - t.recorded_asset_value, 2
        ) AS difference_next
    FROM team_race t
    JOIN roster_values r
      ON r.season = t.season
     AND r.race_number = t.race_number
     AND r.user_guid = t.user_guid
     AND r.team_no = t.team_no
    WHERE t.recorded_asset_value IS NOT NULL
      AND r.asset_count = 7
      AND r.prices_n_found = 7
      AND r.prices_next_found = 7
      AND NOT EXISTS (
          SELECT 1 FROM team_chip_usage c
          WHERE c.season = t.season
            AND c.user_guid = t.user_guid
            AND c.team_no = t.team_no
            AND c.race_used = t.race_number
      )
    ORDER BY t.race_number, t.team_name
"""

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    valuation_comparison = pd.read_sql_query(query, con)

summary = pd.DataFrame([
    {
        "price_feed": label,
        "records_compared": len(valuation_comparison),
        "matching_records": (
            valuation_comparison[column].abs() <= 0.01
        ).sum(),
        "mean_absolute_difference": (
            valuation_comparison[column].abs().mean()
        ),
    }
    for label, column in [
        ("Race N", "difference_n"),
        ("Race N+1", "difference_next"),
    ]
])

summary

,price_feed,records_compared,matching_records,mean_absolute_difference
0,Race N,104,104,0.000000
1,Race N+1,104,1,1.495192


In [2]:
query = """
    SELECT
        t.race_number,
        t.team_name,
        c.display_name AS chip,
        t.recorded_asset_value,
        t.remaining_budget,
        t.recorded_total_budget
    FROM team_race t
    JOIN team_chip_usage u
      ON u.season = t.season
     AND u.user_guid = t.user_guid
     AND u.team_no = t.team_no
     AND u.race_used = t.race_number
    JOIN chips c
      ON c.chip_id = u.chip_id
    ORDER BY c.display_name, t.race_number
"""

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    chip_race_values = pd.read_sql_query(query, con)

chip_race_values

,race_number,team_name,chip,recorded_asset_value,remaining_budget,recorded_total_budget
0,15,Non-za,Auto Pilot,117.5,0.5,118.0
1,15,Gorąco Polecam,Final Fix,109.0,3.8,112.8
2,2,"No Michael, no! That's...",Limitless,99.6,0.1,99.7
3,2,Ramen Face Racing,Limitless,103.1,0.2,103.3
4,4,real and true max fanclub,Limitless,104.1,0.3,104.4
5,5,RossDonalds motorsport,Limitless,106.6,1.0,107.6
6,5,Trulli madly deeply,Limitless,105.4,0.6,106.0
7,6,Leave Briatore alone!!,Limitless,107.8,0.2,108.0
8,6,In Search of Lost Sainz,Limitless,110.5,1.0,111.5
9,12,BBB-TEAM,Limitless,109.0,0.7,109.7


In [3]:
query = """
    SELECT
        t.race_number,
        t.team_name,
        c.display_name AS chip,
        t.recorded_asset_value,
        ROUND(SUM(a.value), 1) AS all_asset_value,
        ROUND(SUM(
            CASE WHEN s.is_final != 1 THEN a.value ELSE 0 END
        ), 1) AS original_roster_value,
        ROUND(SUM(
            CASE WHEN s.is_final != 2 THEN a.value ELSE 0 END
        ), 1) AS replacement_roster_value
    FROM team_race t
    JOIN team_chip_usage u
      ON u.season = t.season
     AND u.user_guid = t.user_guid
     AND u.team_no = t.team_no
     AND u.race_used = t.race_number
    JOIN chips c ON c.chip_id = u.chip_id
    JOIN team_race_asset s
      ON s.season = t.season
     AND s.race_number = t.race_number
     AND s.user_guid = t.user_guid
     AND s.team_no = t.team_no
    JOIN asset_race a
      ON a.season = s.season
     AND a.race_number = s.race_number
     AND a.asset_id = s.asset_id
    GROUP BY
        t.season, t.race_number, t.user_guid, t.team_no,
        t.team_name, c.display_name, t.recorded_asset_value
    ORDER BY c.display_name, t.race_number
"""

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    chip_value_comparison = pd.read_sql_query(query, con)

chip_value_comparison

,race_number,team_name,chip,recorded_asset_value,all_asset_value,original_roster_value,replacement_roster_value
0,15,Non-za,Auto Pilot,117.5,117.5,117.5,117.5
1,15,Gorąco Polecam,Final Fix,109.0,121.0,109.0,112.6
2,2,"No Michael, no! That's...",Limitless,99.6,178.1,178.1,178.1
3,2,Ramen Face Racing,Limitless,103.1,177.2,177.2,177.2
4,4,real and true max fanclub,Limitless,104.1,187.7,187.7,187.7
5,5,RossDonalds motorsport,Limitless,106.6,187.2,187.2,187.2
6,5,Trulli madly deeply,Limitless,105.4,191.8,191.8,191.8
7,6,In Search of Lost Sainz,Limitless,110.5,184.9,184.9,184.9
8,6,Leave Briatore alone!!,Limitless,107.8,184.9,184.9,184.9
9,12,BBB-TEAM,Limitless,109.0,191.6,191.6,191.6


## Testing Marts

In [4]:
with sqlite3.connect("../../data/f1_fantasy.db") as con:
    preview = pd.read_sql_query("""
        SELECT *
        FROM mart_team_race
        ORDER BY user_guid, team_no, race_number
        LIMIT 15
    """, con)

preview

,season,race_number,race_name,user_guid,team_no,team_name,race_points,race_rank,cumulative_points,league_average_race_points,recorded_asset_value,remaining_budget,recorded_total_budget
0,2026,1,Australian Grand Prix,00190f2c-133c-11f1-9186-ede0d682ac23-0-231965328,1,In Search of Lost Sainz,196.0,7,196.0,180.13,97.9,2.1,100.0
1,2026,2,Chinese Grand Prix,00190f2c-133c-11f1-9186-ede0d682ac23-0-231965328,1,In Search of Lost Sainz,398.0,5,594.0,363.53,100.0,2.7,102.7
2,2026,3,Japanese Grand Prix,00190f2c-133c-11f1-9186-ede0d682ac23-0-231965328,1,In Search of Lost Sainz,207.0,3,801.0,183.40,103.7,1.1,104.8
3,2026,4,Miami Grand Prix,00190f2c-133c-11f1-9186-ede0d682ac23-0-231965328,1,In Search of Lost Sainz,199.0,8,1000.0,197.93,NaN,1.1,NaN
4,2026,5,Canadian Grand Prix,00190f2c-133c-11f1-9186-ede0d682ac23-0-231965328,1,In Search of Lost Sainz,294.0,4,1294.0,237.27,110.0,1.0,111.0
5,2026,6,Monaco Grand Prix,00190f2c-133c-11f1-9186-ede0d682ac23-0-231965328,1,In Search of Lost Sainz,144.0,8,1438.0,136.07,110.5,1.0,111.5
6,2026,7,Barcelona-Catalunya Grand Prix,00190f2c-133c-11f1-9186-ede0d682ac23-0-231965328,1,In Search of Lost Sainz,112.0,6,1550.0,110.07,109.6,3.2,112.8
7,2026,8,Austrian Grand Prix,00190f2c-133c-11f1-9186-ede0d682ac23-0-231965328,1,In Search of Lost Sainz,244.0,1,1794.0,189.60,114.3,0.6,114.9
8,2026,9,British Grand Prix,00190f2c-133c-11f1-9186-ede0d682ac23-0-231965328,1,In Search of Lost Sainz,309.0,2,2103.0,268.07,NaN,0.6,NaN
9,2026,10,Belgian Grand Prix,00190f2c-133c-11f1-9186-ede0d682ac23-0-231965328,1,In Search of Lost Sainz,243.0,2,2346.0,202.33,NaN,0.6,NaN


In [5]:
with sqlite3.connect("../../data/f1_fantasy.db") as con:
    summary = pd.read_sql_query("""
        SELECT
            valuation_source,
            COUNT(*) AS rows,
            SUM(total_wealth IS NULL) AS missing_wealth
        FROM mart_team_race
        GROUP BY valuation_source
    """, con)

summary

,valuation_source,rows,missing_wealth
0,calculated,97,0
1,recorded,128,0


In [6]:
with sqlite3.connect("../../data/f1_fantasy.db") as con:
    mart = pd.read_sql_query("SELECT * FROM mart_team_race", con)

pd.Series({
    "rows": len(mart),
    "duplicate_keys": mart.duplicated(
        ["season", "race_number", "user_guid", "team_no"]
    ).sum(),
    "missing_wealth": mart["total_wealth"].isna().sum(),
    "missing_wealth_change_after_race_1": mart.loc[
        mart["race_number"] > 1, "wealth_change"
    ].isna().sum(),
})

rows                                  239
duplicate_keys                          0
missing_wealth                          0
missing_wealth_change_after_race_1      0
dtype: int64

In [7]:
with sqlite3.connect("../../data/f1_fantasy.db") as con:
    chips = pd.read_sql_query("SELECT * FROM mart_team_chips", con)

display(chips.groupby("chip_status").size().rename("rows"))

print("Total rows:", len(chips))
print(
    "Duplicate keys:",
    chips.duplicated(
        ["season", "user_guid", "team_no", "chip_id"]
    ).sum()
)

display(chips[chips["team_name"] == "Gogo@interlagos"])

chip_status
unused    65
used      25
Name: rows, dtype: int64

Total rows: 90
Duplicate keys: 0


,season,user_guid,team_no,team_name,last_observed_race,chip_id,chip_name,available_first_race,race_used,chip_status,race_used_name,chip_race_points,league_average_race_points
0,2026,491f6d8c-19aa-11f1-8f1b-c30b1cdb633b-0-218923183,1,Gogo@interlagos,15,1,Limitless,2,NaN,unused,None,NaN,NaN
1,2026,491f6d8c-19aa-11f1-8f1b-c30b1cdb633b-0-218923183,1,Gogo@interlagos,15,2,Wildcard,2,NaN,unused,None,NaN,NaN
2,2026,491f6d8c-19aa-11f1-8f1b-c30b1cdb633b-0-218923183,1,Gogo@interlagos,15,3,Final Fix,2,NaN,unused,None,NaN,NaN
3,2026,491f6d8c-19aa-11f1-8f1b-c30b1cdb633b-0-218923183,1,Gogo@interlagos,15,4,Auto Pilot,1,NaN,unused,None,NaN,NaN
4,2026,491f6d8c-19aa-11f1-8f1b-c30b1cdb633b-0-218923183,1,Gogo@interlagos,15,5,No Negative,1,NaN,unused,None,NaN,NaN
5,2026,491f6d8c-19aa-11f1-8f1b-c30b1cdb633b-0-218923183,1,Gogo@interlagos,15,6,x3 Boost,1,NaN,unused,None,NaN,NaN
